In [1]:
import gc
gc.collect()

import sys
sys.path.insert(0, '../')
import logging
logging.basicConfig(level=logging.ERROR)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from cryptotrader.exchange_api.poloniex import Poloniex
from cryptotrader.envs.trading import BacktestDataFeed, BacktestEnvironment
from cryptotrader.envs.utils import make_balance, convert_to
from cryptotrader.agents import apriori
from cryptotrader.utils import array_normalize

from time import time, sleep

from bokeh.io import output_notebook
output_notebook()
%matplotlib inline
# %load_ext line_profiler

Loading BokehJS ...

In [ ]:
# Environment setup
obs_steps = 250
period = 30
pairs = ["USDT_BTC", "USDT_ETH", "USDT_LTC", "USDT_XRP", "USDT_XMR", "USDT_ETC", "USDT_DASH"]
init_funds = make_balance(crypto=0.0, fiat=100.0, pairs=pairs)

tapi = Poloniex()
tapi = BacktestDataFeed(tapi, period, pairs, init_funds)
tapi.download_data(end=datetime.timestamp(datetime.utcnow() - timedelta(days=200)),
                       start=datetime.timestamp(datetime.utcnow() - timedelta(days=300)))

# tapi.save_data('./')
env = BacktestEnvironment(period, obs_steps, tapi, 'backtest_env')
env.add_pairs(pairs)
env.fiat = 'USDT'
obs = env.reset()

INFO:backtest_env:[Trading Environment initialization]
Trading Environment Initialized!



In [ ]:
# Training params
nb_steps = 300
batch_size = 4
nb_max_episode_steps = 12
# {'variant': 'PAMR1', 'C': 2444.4914762280923, 'sensitivity': 0.031091386721483268} 
agent = apriori.TCOTrader()

search_space={
                'window':[3, 200],
                'smooth':[0, 4],
                'toff':[0, 0.5]
                }

opt_params, info = agent.fit(env, nb_steps, batch_size, search_space, nb_max_episode_steps=nb_max_episode_steps)
print("\n", opt_params,"\n", env.status)

agent.test(env, verbose=True)
env.plot_results();

Optimizing model for 200 steps with batch size 4...


In [ ]:
tapi.download_data(end=datetime.timestamp(datetime.utcnow() - timedelta(days=100)),
                       start=datetime.timestamp(datetime.now() - timedelta(days=200)))

agent.test(env, verbose=True)
env.plot_results();

In [ ]:
tapi.download_data(end=datetime.timestamp(datetime.utcnow()),
                       start=datetime.timestamp(datetime.now() - timedelta(days=100)))

agent.test(env, verbose=True)
env.plot_results();